# Exercise 2 — A latent space you can see ($d = 2$)

Same VAE and training budget as Exercise 1, with $d = 2$. Scatter the encoded
test set, decode a prior-mass grid, and compare pixel vs latent interpolation
(latent row uses the $d = 16$ model).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import norm
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from vae import VAE, evaluate, train_vae

%matplotlib inline

FIG_DIR = Path("..") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DEVICE = torch.device("cpu")

## A — Train a 2-D VAE

In [ ]:
train_full = datasets.MNIST(DATA_DIR, train=True, download=True,
                            transform=transforms.ToTensor())
test = datasets.MNIST(DATA_DIR, train=False, download=True,
                      transform=transforms.ToTensor())
train, val = random_split(train_full, [50_000, 10_000],
                          generator=torch.Generator().manual_seed(42))

torch.manual_seed(42)
model2 = VAE(latent_dim=2)
hist2 = train_vae(model2, train, val, epochs=20, batch_size=128, lr=1e-3,
                  device=DEVICE)

test_loader = DataLoader(test, batch_size=256, shuffle=False)
test_recon2, test_kl2 = evaluate(model2, test_loader, DEVICE)
print(f"d=2  test recon={test_recon2:.2f}  kl={test_kl2:.2f}")

# Compare with the d=16 checkpoint from Exercise 1
model16 = VAE(latent_dim=16)
model16.load_state_dict(torch.load("vae_d16.pt", map_location="cpu", weights_only=True))
model16.eval()
test_recon16, test_kl16 = evaluate(model16, test_loader, DEVICE)
print(f"d=16 test recon={test_recon16:.2f}  kl={test_kl16:.2f}")

## B — The encoded test set

In [ ]:
model2.eval()
mus, labels = [], []
with torch.no_grad():
    for x, y in DataLoader(test, batch_size=512, shuffle=False):
        mu, _ = model2.encode(x.view(x.size(0), -1))
        mus.append(mu)
        labels.append(y)
mu = torch.cat(mus).numpy()
y = torch.cat(labels).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
for d in range(10):
    m = y == d
    ax.scatter(mu[m, 0], mu[m, 1], s=6, alpha=0.45, label=str(d))
ax.set_title(r"Figure 4 — Encoded test set ($\mu$, $d=2$)")
ax.set_xlabel(r"$\mu_1$")
ax.set_ylabel(r"$\mu_2$")
ax.legend(title="digit", markerscale=3, loc="best")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_latent_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## C — The decoded plane

Grid points use $z = \Phi^{-1}(q)$ with $q$ evenly spaced in $[0.05, 0.95]$, so
each cell has the same prior mass under $\mathcal{N}(0, I)$.

In [ ]:
qs = np.linspace(0.05, 0.95, 15)
coords = norm.ppf(qs)
grid = np.stack(np.meshgrid(coords, coords[::-1]), axis=-1).reshape(-1, 2)

with torch.no_grad():
    z = torch.tensor(grid, dtype=torch.float32)
    imgs = model2.decode(z).view(-1, 28, 28).numpy()

canvas = np.zeros((15 * 28, 15 * 28))
for i in range(15):
    for j in range(15):
        canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = imgs[i * 15 + j]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(canvas, cmap="gray")
ax.set_title(r"Figure 5 — Decoded $15\times 15$ plane ($z=\Phi^{-1}(q)$)")
ax.set_xlabel(r"$z_1$ (quantile grid)")
ax.set_ylabel(r"$z_2$ (quantile grid)")
ax.set_xticks([])
ax.set_yticks([])
fig.tight_layout()
fig.savefig(FIG_DIR / "fig5_decoded_plane.png", dpi=150, bbox_inches="tight")
plt.show()

## D — Interpolation

Top row: linear blend in pixel space. Bottom row: decode a linear blend of
$\mu$ with the $d = 16$ model from Exercise 1.

In [ ]:
def first_index(digit):
    for i in range(len(test)):
        if int(test[i][1]) == digit:
            return i
    raise ValueError(digit)

idx_a, idx_b = first_index(4), first_index(9)
xa, ya = test[idx_a]
xb, yb = test[idx_b]
print(f"indices {idx_a} (digit {int(ya)}) → {idx_b} (digit {int(yb)})")

with torch.no_grad():
    mu_a, _ = model16.encode(xa.view(1, -1))
    mu_b, _ = model16.encode(xb.view(1, -1))

ts = np.linspace(0, 1, 10)
fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for k, t in enumerate(ts):
    blend = ((1 - t) * xa + t * xb).view(28, 28)
    axes[0, k].imshow(blend, cmap="gray")
    axes[0, k].axis("off")
    axes[0, k].set_title(f"$t={t:.1f}$", fontsize=8)
    with torch.no_grad():
        img = model16.decode((1 - t) * mu_a + t * mu_b).view(28, 28).numpy()
    axes[1, k].imshow(img, cmap="gray")
    axes[1, k].axis("off")
axes[0, 0].set_ylabel("pixel", fontsize=9)
axes[1, 0].set_ylabel("latent", fontsize=9)
fig.suptitle(f"Figure 6 — Interpolation (indices {idx_a} → {idx_b})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig6_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

torch.save(model2.state_dict(), "vae_d2.pt")